# 06 - Ranking Gate: the Agreement Gate, the Fine Family Key, and the Formula

**What this notebook is for.** It is the decision record for **run 3** (`20260911T121013Z`) of the
ranking selection experiment, and for the decisions it produced: **Q29** (the formula) and **Q30**
(the agreement gate). Notebook 05 records runs 1-2; this one picks up where 05's trace left off.

**Where run 2 left things.** Run 2's trace (notebook 05, F2-F3) found two failure modes, both caused by
a family learning from too little evidence:

1. **Family collision.** One *correct* dismissal of a benign flow the model scored 99.89 as a Web
   Attack demoted 59 true Web Attacks in the same coarse family.
2. **Tier 2 flood.** Under movement M2, one *wrong* confirmation outweighed thirteen correct
   dismissals of a benign DNS family and lifted its future flows into the Tier 2 band.

**What run 3 added**, as two new arm dimensions:

- the collaborator's **agreement gate** (`stage-5/config/adaptation-config.json`, `aggregation`): a
  family's learning reaches the queue only after at least 3 learning verdicts, no tie, and a dominant
  direction holding at least 0.67 of them - and then only the learning that points that way;
- a **fine family key**, which adds the destination IP for every flow.

The selection rule, **sel-3**, was committed before the run. Every number below is computed from the
run's own `results.json` or by re-running the experiment's own code; nothing is typed in by hand.

In [1]:
import json, platform, subprocess, sys
from collections import Counter
from pathlib import Path
import pandas as pd

here = Path.cwd()
REPO = next(p for p in [here, *here.parents] if (p / "hitl-ids/data/processed").exists())
HITL = REPO / "hitl-ids"
sys.path.insert(0, str(HITL))
RANK = HITL / "evaluation/ranking"

# Run 3 is named explicitly, so appending later runs never changes what this notebook records.
RUN_ID = "20260911T121013Z"
PREVIOUS_ID = "20260911T111625Z"   # run 2, analysed in notebook 05
RUN = RANK / "runs" / RUN_ID

history = [json.loads(line) for line in
           (RANK / "history.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
config = json.loads((RUN / "config.json").read_text(encoding="utf-8"))
results = json.loads((RUN / "results.json").read_text(encoding="utf-8"))
previous = json.loads((RANK / "runs" / PREVIOUS_ID / "results.json").read_text(encoding="utf-8"))
method_md = (RUN / "METHOD.md").read_text(encoding="utf-8")
entry = next(e for e in history if e["run_id"] == RUN_ID)

print(f"runs in history : {[e['run_id'] for e in history]}")
print(f"this record     : {RUN_ID}  (selection rule {entry['selection_rule']}, commit {entry['commit']})")
print(f"arms recorded   : {len(results['runs'])}   aggregates: {len(results['aggregates'])}")

runs in history : ['20260911T111249Z', '20260911T111625Z', '20260911T121013Z']
this record     : 20260911T121013Z  (selection rule sel-3, commit 5269732)
arms recorded   : 576   aggregates: 192


---
## 1. Method

The method file is written by the experiment itself into the run folder, so this is the run's own
account of what it did.

In [2]:
from IPython.display import Markdown, display

display(Markdown(method_md))

# Ranking selection experiment — method

Design: `docs/ranking-and-escalation-design.md` §6. Code: `packages/detection/ranking/`.

1. **Data.** All 5,000 demo flows, fused by the production engine (S5 rules + S6 fusion), ordered by
   timestamp. First half = calibration (the past); second half = future (never shown to the analyst).
2. **Families.** *Coarse*: predicted class + destination port + protocol + matched rule; unflagged
   flows add the destination IP. *Fine*: every flow adds the destination IP. A verdict updates its
   family; future members inherit the family's state.
3. **Simulated Tier 1 analyst.** 10 rounds; each round reviews the top 25 unreviewed
   calibration alerts plus 5 randomly sampled unflagged ones (QA sampling). Verdict =
   ground truth, flipped with probability 0 %, 5 % or 15 % (analyst error).
4. **Arms.** Formulas C0-C3 x movement M1/M2 x guardrails on/off x agreement gate on/off x family
   key coarse/fine x error rate x 3 seeds.
   Guardrails off removes the caps, floors, I3 and tier-loss protection; the 0-100 range remains.
   The gate (the collaborator's aggregation rule) applies a family's learning only once it has at
   least 3 learning verdicts, no tie, and a dominant direction holding at least
   0.67 of them - and then only the learning that points that way.
5. **Metrics, future half only**: precision in the top 50/100/200; mean attack position (0-1, lower
   is better); last attack position; benign alerts in the top 100; Critical floor violations;
   true attacks demoted; Tier 2 load and precision; class changes during calibration.
6. **Selection (sel-3), fixed in code before the run.** Candidates: guarded, gated arms.
   Ranked by: the step depends on the attack type's severity (Q27 - C0 does not); safety (no floor
   violation at any error rate, no attack demoted at 0 % error); fewest true attacks demoted under
   analyst error (5 % + 15 %); highest Tier 2 precision at 5 %; lowest mean attack position at 5 %;
   fewest class changes; then the simpler formula, movement and family key.
   History: *sel-1* (run 20260911T111249Z) could not discriminate - the control already scores 1.0
   in the top 100. *sel-2* (run 20260911T111625Z) was written after run 1, and its deciding metric
   traced to one family collision (changelog v1.12).


In [3]:
print(pd.Series({
    "flows (demo sample, timestamp order)": config["flows"],
    "calibration half (the analyst sees these)": config["calibration"],
    "future half (scored, never shown)": config["future"],
    "attacks in calibration": config["calibration_attacks"],
    "attacks in future": config["future_attacks"],
    "seeds": config["seeds"],
    "error rates": config["error_rates"],
    "family keys": config["family_keys"],
    "gate": f"min {config['gate']['min_feedback']} verdicts, agreement >= {config['gate']['min_agreement']}",
    "severity chart version": config["severity_chart"],
    "selection rule": config["selection_rule"],
    "recorded environment": config["environment"],
}).to_string())

flows (demo sample, timestamp order)                                                      5000
calibration half (the analyst sees these)                                                 2500
future half (scored, never shown)                                                         2500
attacks in calibration                                                                     600
attacks in future                                                                          400
seeds                                                           [20260911, 20260912, 20260913]
error rates                                                                  [0.0, 0.05, 0.15]
family keys                                                                     [coarse, fine]
gate                                                         min 3 verdicts, agreement >= 0.67
severity chart version                                                                   sev-1
selection rule                                    

**sel-3 was fixed before the run.** The run records the commit it ran from. The cell below reads the
selection rule's name out of that commit with git, so the claim rests on the repository, not on this
text.

In [4]:
def at_commit(path):
    try:
        return subprocess.run(["git", "-C", str(HITL), "show", f"{config['commit']}:{path}"],
                              capture_output=True, text=True, check=True).stdout
    except (OSError, subprocess.CalledProcessError) as error:
        return f"<git unavailable: {error}>"

source = at_commit("hitl-ids/packages/detection/ranking/experiment.py")
rule_line = next((line for line in source.splitlines() if line.startswith("SELECTION_RULE")),
                 "<not found>")
print(f"commit {config['commit']}: {rule_line}")
log = subprocess.run(["git", "-C", str(HITL), "log", "-1", "--format=%h %cI %s", config["commit"]],
                     capture_output=True, text=True).stdout.strip()
print(f"commit {log}")
print(f"run started   {config['started']}")

commit 5269732: SELECTION_RULE = "sel-3"
commit 5269732 2026-09-11T20:10:12+08:00 Ranking: agreement gate and fine family key; selection rule sel-3 fixed before run 3
run started   2026-09-11T12:10:13.689381Z


**Interpretation.** The run's commit, `5269732`, already carries `SELECTION_RULE = "sel-3"`, and git dates that commit
(20:10:12 +08:00, i.e. 12:10:12 UTC) before the run's recorded start (12:10:13.69 UTC). The rule was fixed
before any run-3 number existed. It is the third selection rule and the first written before its run;
notebook 05 records why sel-1 and sel-2 were replaced.

---
## 2. Run 3 reproduces run 2

Run 3's *ungated, coarse* arms are run 2's arms under a new name. If the code change that added the
gate and the fine key had disturbed anything else, these would differ.

In [5]:
ARM = ["formula", "movement", "guardrails", "error_rate"]
metric_names = [k for k in previous["aggregates"][0] if k not in ARM + ["seeds"]]
run2 = {tuple(a[k] for k in ARM): a for a in previous["aggregates"]}
run3 = {tuple(a[k] for k in ARM): a for a in results["aggregates"]
        if not a["gated"] and a["family_key"] == "coarse"}
same = [key for key in run2 if key in run3
        and all(run2[key][name] == run3[key][name] for name in metric_names)]
print(f"run-2 aggregates: {len(run2)}   run-3 ungated coarse aggregates: {len(run3)}")
print(f"identical in all {len(metric_names)} run-2 metrics: {len(same)} of {len(run2)}")

run-2 aggregates: 48   run-3 ungated coarse aggregates: 48
identical in all 14 run-2 metrics: 48 of 48


The traces in sections 4-5 re-run the calibration phase with the experiment's own code. They mean
nothing unless this kernel replays the recorded run exactly, so every recorded gated, guarded arm at
5 % error is re-simulated first, and the notebook stops if any differs.

In [6]:
import pandas, numpy
from packages.detection.ranking import experiment as E
from packages.detection.ranking import formulas as F
from packages.detection.ranking.severity import load_severity_chart

chart = load_severity_chart(HITL / config["severity_chart_file"])
assert chart.version == config["severity_chart"], (chart.version, config["severity_chart"])
flows = {key: E.load_flows(HITL / "data/processed", chart, key) for key in config["family_keys"]}
half = config["calibration"]
split = {key: (keyed[:half], keyed[half:]) for key, keyed in flows.items()}
kw = dict(rounds=config["rounds"], batch=config["batch"], qa_sample=config["qa_sample"])


def arm_of(r):
    return E.Arm(r["formula"], r["movement"], r["guardrails"], r["error_rate"], r["seed"],
                 r["gated"], r["family_key"])


probes = [r for r in results["runs"] if r["guardrails"] and r["gated"] and r["error_rate"] == 0.05]
mismatched = [arm_of(r) for r in probes
              if E.simulate(*split[r["family_key"]], arm_of(r), **kw) != r]
assert not mismatched, f"this kernel does not replay run {RUN_ID}: {mismatched}"
print(f"{len(probes)} recorded gated arms replay identically "
      f"(python {platform.python_version()}, pandas {pandas.__version__}, numpy {numpy.__version__}; "
      f"recorded under {config['environment']})")

48 recorded gated arms replay identically (python 3.12.6, pandas 3.0.5, numpy 2.5.2; recorded under {'python': '3.11.11', 'pandas': '2.3.3', 'numpy': '2.3.5'})


---
## 3. Results

Guarded arms only (the design S7b builds). "Attacks demoted" sums the seed means at 5 % and 15 %
analyst error, as sel-3 does; the other columns are at 5 % error.

In [7]:
agg = pd.DataFrame(results["aggregates"])
guarded = agg[agg.guardrails].copy()
guarded["setting"] = (guarded.gated.map({False: "ungated", True: "gated"}) + ", "
                      + guarded.family_key)
guarded["arm"] = guarded.formula + "+" + guarded.movement

demoted = (guarded[guarded.error_rate > 0].groupby(["setting", "arm"]).attacks_demoted.sum()
           .round(4).rename("attacks demoted (5%+15%)"))
at5 = guarded[guarded.error_rate == 0.05].set_index(["setting", "arm"])[
    ["tier2_precision", "tier2_load", "mean_attack_position"]]
table = at5.join(demoted)
with pd.option_context("display.width", 140):
    print(table.to_string())
print(f"\ncontrol (no feedback): {results['control']}")

                       tier2_precision  tier2_load  mean_attack_position  attacks demoted (5%+15%)
setting         arm                                                                               
ungated, coarse C0+M1           1.0000    223.3333                0.0828                   78.6667
ungated, fine   C0+M1           1.0000    243.0000                0.0828                    0.0000
gated, coarse   C0+M1           1.0000    243.0000                0.0828                    0.0000
gated, fine     C0+M1           1.0000    243.0000                0.0828                    0.0000
ungated, coarse C0+M2           0.7405    484.3333                0.1236                   78.6667
ungated, fine   C0+M2           0.7418    504.0000                0.1235                    0.0000
gated, coarse   C0+M2           1.0000    243.0000                0.0828                    0.0000
gated, fine     C0+M2           1.0000    243.0000                0.0828                    0.0000
ungated, c

In [8]:
summary = table.groupby(level="setting").agg(["min", "max"])
with pd.option_context("display.width", 160):
    print("range over the eight formula+movement arms, per setting:\n")
    print(summary.to_string())
print(f"\nCritical floor violations, any guarded arm, any error rate: "
      f"{int(guarded.floor_violations.max())}")
print(f"true attacks demoted at 0 % error, any guarded arm: "
      f"{guarded[guarded.error_rate == 0].attacks_demoted.max()}")

range over the eight formula+movement arms, per setting:

                tier2_precision      tier2_load           mean_attack_position         attacks demoted (5%+15%)       
                            min  max        min       max                  min     max                      min    max
setting                                                                                                               
gated, coarse            1.0000  1.0      243.0  243.0000               0.0828  0.0829                      0.0    0.0
gated, fine              1.0000  1.0      243.0  243.0000               0.0828  0.0829                      0.0    0.0
ungated, coarse          0.7253  1.0      184.0  504.0000               0.0828  0.2004                      0.0  118.0
ungated, fine            0.7400  1.0      243.0  530.3333               0.0828  0.1682                      0.0    0.0

Critical floor violations, any guarded arm, any error rate: 0
true attacks demoted at 0 % error, any guarded

**Interpretation.**

- **Gated, either key:** all eight formula+movement arms match the control - Tier 2 precision 1.00,
  load 243, mean attack position 0.0828-0.0829, and no attack demoted under analyst error.
- **Ungated:** under the coarse key run 2's failures reappear - up to 118 attacks demoted (C2/C3), and
  under M2 a Tier 2 precision as low as 0.7253 and a load up to 504. The fine key removes the
  demotions (0 for every arm) but not the M2 load (up to 530.33, precision 0.7400).
- **Safety held in every guarded arm:** no Critical floor violation at any error rate, no attack
  demoted at 0 % error.

The gated rows equal the control on every column. Section 4 asks whether that is protection or
inactivity.

---
## 4. Findings

Each finding is computed in a code cell, then interpreted. The helpers re-run calibration with the
experiment's own functions (`calibrate`, `rank`, `baseline_class`), exactly as `simulate()` does.

In [9]:
SEED = config["seeds"][0]


def trace(formula, movement, error_rate, seed, gated, key="coarse"):
    calibration, future = split[key]
    families, log = E.calibrate(calibration, E.Arm(formula, movement, True, error_rate, seed,
                                                   gated, key), **kw)
    return families, log, E.rank(future, families, True, gated)


def verdicts(log, family):
    return dict(Counter((e["verdict"], "wrong" if e["wrong"] else "correct",
                         "attack" if e["malicious"] else "benign")
                        for e in log if e["family"] == family))


def demoted_by_family(queue):
    return Counter(f.family for cls, score, f in queue if f.malicious and (
        score < f.detection_score or cls > E.baseline_class(f)))


def benign_tier2_by_family(queue):
    return Counter(f.family for cls, score, f in queue if cls == 0 and not f.malicious)


def gate_open(state):
    direction, share = F.agreement(state)
    return (state.confirmations + state.dismissals >= F.GATE_MIN_FEEDBACK and direction != 0
            and share >= F.GATE_MIN_AGREEMENT)


WEB = next(f for f in flows["coarse"] if f.alert_id == "AL-00478").family
print(f"the colliding family from run 2 (holds AL-00478): {WEB}")

the colliding family from run 2 (holds AL-00478): ('Web Attack', 80, 'TCP', '-')


### F1 - the gate removes both of run 2's failure modes

In [10]:
rows = []
for seed in config["seeds"]:
    for formula in ("C0", "C2"):
        for gated in (False, True):
            families, log, queue = trace(formula, "M1", 0.05, seed, gated)
            state = families.get(WEB)
            rows.append({
                "seed": seed, "formula": formula, "gate": "on" if gated else "off",
                "attacks demoted": sum(demoted_by_family(queue).values()),
                "of which in WEB": demoted_by_family(queue)[WEB],
                "WEB verdicts": verdicts(log, WEB) or "-",
                "WEB gate open": gate_open(state) if state else "-",
            })
print("failure mode 1, the collision - movement M1, 5 % error, guardrails on:\n")
with pd.option_context("display.width", 200, "display.max_colwidth", 90):
    print(pd.DataFrame(rows).to_string(index=False))

failure mode 1, the collision - movement M1, 5 % error, guardrails on:

    seed formula gate  attacks demoted  of which in WEB                                      WEB verdicts WEB gate open
20260911      C0  off               59               59 {('mark_false_positive', 'correct', 'benign'): 1}         False
20260911      C0   on                0                0                                                 -             -
20260911      C2  off               59               59 {('mark_false_positive', 'correct', 'benign'): 1}         False
20260911      C2   on                0                0                                                 -             -
20260912      C0  off                0                0                                                 -             -
20260912      C0   on                0                0                                                 -             -
20260912      C2  off               59               59 {('mark_false_positive', 'correc

In [11]:
rows, flooded = [], []
for seed in config["seeds"]:
    for gated in (False, True):
        families, log, queue = trace("C1", "M2", 0.05, seed, gated)
        benign = benign_tier2_by_family(queue)
        top = benign.most_common(1)[0][0] if benign else None
        rows.append({"seed": seed, "gate": "on" if gated else "off",
                     "benign alerts in Tier 2": sum(benign.values()),
                     "largest benign family": top or "-",
                     "its verdicts": verdicts(log, top) if top else "-"})
        if not gated and top:
            flooded.append((seed, top))
print("failure mode 2, the M2 flood - C1 + M2, 5 % error, guardrails on:\n")
with pd.option_context("display.width", 220, "display.max_colwidth", 90):
    print(pd.DataFrame(rows).to_string(index=False))

print("\nthe same flooded families with the gate on - how their state is gated:")
for seed, family in flooded:
    families, _, _ = trace("C1", "M2", 0.05, seed, True)
    state = families[family]
    print(f"  seed {seed} {family}: confirmations {state.confirmations}, dismissals "
          f"{state.dismissals}, agreement {F.agreement(state)}, gate open {gate_open(state)}; "
          f"learned (adjustment, offset) = ({round(state.adjustment, 2)}, {state.class_offset}) "
          f"-> applied {tuple(round(x, 2) for x in F.effective(state, True))}")

failure mode 2, the M2 flood - C1 + M2, 5 % error, guardrails on:

    seed gate  benign alerts in Tier 2                largest benign family                                                                                        its verdicts
20260911  off                        3 (Benign, 3389, TCP, -, 172.31.67.58)                                                   {('confirm_true_positive', 'wrong', 'benign'): 1}
20260911   on                        0                                    -                                                                                                   -
20260912  off                        0                                    -                                                                                                   -
20260912   on                        0                                    -                                                                                                   -
20260913  off                      780     (Benign, 5

**Isolating the gate from queue order.** A gated arm's analyst works a differently ordered queue, so
the tables above mix two effects: the gate withholding learning, and the analyst reviewing different
flows. The cell below removes the second. It takes the family states the **ungated** arms actually
learned - the very verdicts that caused run 2's failures - and scores the future half twice, once
without and once with the gate.

In [12]:
rows = []
for formula, movement, measure in (("C0", "M1", "demoted"), ("C2", "M1", "demoted"),
                                   ("C1", "M2", "benign in Tier 2")):
    for seed in config["seeds"]:
        families, _, _ = trace(formula, movement, 0.05, seed, gated=False)
        count = {}
        for gated in (False, True):
            queue = E.rank(split["coarse"][1], families, True, gated)
            count[gated] = sum((demoted_by_family(queue) if measure == "demoted"
                                else benign_tier2_by_family(queue)).values())
        rows.append({"arm": f"{formula}+{movement}", "seed": seed, "measure": measure,
                     "ungated verdicts, scored ungated": count[False],
                     "same verdicts, scored gated": count[True]})
print("the ungated arms' own learned families, scored without and with the gate "
      "(5 % error, guardrails on, coarse key):\n")
print(pd.DataFrame(rows).to_string(index=False))

the ungated arms' own learned families, scored without and with the gate (5 % error, guardrails on, coarse key):

  arm     seed          measure  ungated verdicts, scored ungated  same verdicts, scored gated
C0+M1 20260911          demoted                                59                            0
C0+M1 20260912          demoted                                 0                            0
C0+M1 20260913          demoted                                 0                            0
C2+M1 20260911          demoted                                59                            0
C2+M1 20260912          demoted                                59                            0
C2+M1 20260913          demoted                                59                            0
C1+M2 20260911 benign in Tier 2                                 3                            0
C1+M2 20260912 benign in Tier 2                                 0                            0
C1+M2 20260913 benign in Tier 2

**Interpretation.**

- **Without the gate**, both failures reappear as run 2 recorded them.
  - The collision demotes 59 attacks, all in `('Web Attack', 80, 'TCP', '-')`, in the C2 arm of every
    seed and the C0 arm of one. Each time the cause is one *correct* dismissal of a benign flow.
  - The M2 flood puts 780 benign alerts into Tier 2 in seed 20260913, where the DNS family received 13
    correct dismissals and 1 wrong confirmation. Seed 20260911 adds 3; seed 20260912 adds none.
- **With the gate, both counts are 0 - but the first two tables overstate the gate's part.** In the gated
  arms the analyst never reviewed the Web Attack family, nor the RDP flow behind seed 20260911's 3 benign
  alerts: the gated queue is ordered differently, so those flows were never reached.
- **The counterfactual settles it.** Scoring the ungated arms' *own* learned families - the verdicts that
  caused the failures - with the gate on gives 0 demoted attacks and 0 benign Tier 2 alerts in every
  seed. The gate alone is sufficient.
- **The DNS family shows the mechanism.** With the gate on it received 19 correct dismissals and 1 wrong
  confirmation (agreement 0.95). Its gate opened in the dismissing direction, so the learned -30 applies,
  and the promotion the wrong confirmation earned (offset -4) is withheld.

### F2 - the fine key alone removes the collision, but not M2's flood

In [13]:
fine_of = {f.alert_id: f.family for f in flows["fine"]}
future_web = [f for f in split["coarse"][1] if f.family == WEB]
print(f"AL-00478   coarse family {WEB}")
print(f"           fine family   {fine_of['AL-00478']}")
print(f"\nfuture flows in the coarse family: {len(future_web)}, "
      f"true attacks {sum(f.malicious for f in future_web)}")
by_ip = Counter((fine_of[f.alert_id][-1], "attack" if f.malicious else "benign") for f in future_web)
print("split by destination IP (the fine key):")
for (ip, kind), n in by_ip.most_common():
    print(f"   {ip:<16} {kind:<7} {n}")

AL-00478   coarse family ('Web Attack', 80, 'TCP', '-')
           fine family   ('Web Attack', 80, 'TCP', '-', '64.150.178.87')

future flows in the coarse family: 59, true attacks 59
split by destination IP (the fine key):
   172.31.69.28     attack  59


In [14]:
fine_ungated = guarded[(~guarded.gated) & (guarded.family_key == "fine") & (guarded.error_rate > 0)]
print("ungated, fine key - attacks demoted, and Tier 2 at each error rate (seed means):\n")
print(fine_ungated.pivot_table(index="arm", columns="error_rate",
                               values=["attacks_demoted", "tier2_load", "tier2_precision"])
      .round(4).to_string())

ungated, fine key - attacks demoted, and Tier 2 at each error rate (seed means):

           attacks_demoted      tier2_load           tier2_precision        
error_rate            0.05 0.15       0.05      0.15            0.05    0.15
arm                                                                         
C0+M1                  0.0  0.0   243.0000  243.0000          1.0000  1.0000
C0+M2                  0.0  0.0   504.0000  819.6667          0.7418  0.4782
C1+M1                  0.0  0.0   243.0000  243.0000          1.0000  1.0000
C1+M2                  0.0  0.0   504.0000  819.6667          0.7418  0.4782
C2+M1                  0.0  0.0   243.0000  243.0000          1.0000  1.0000
C2+M2                  0.0  0.0   530.3333  819.6667          0.7400  0.4769
C3+M1                  0.0  0.0   243.0000  243.0000          1.0000  1.0000
C3+M2                  0.0  0.0   530.3333  819.6667          0.7400  0.4769


**Interpretation.**

- The fine key separates exactly the two groups that collided: AL-00478, the benign flow, went to
  64.150.178.87; all 59 true attacks in the coarse family's future went to 172.31.69.28.
- Under the fine key no ungated arm demotes an attack, at either error rate.
- It does nothing for M2. Ungated, M2's Tier 2 load is 504-530 at 5 % error and 819.67 at 15 %, with
  precision falling to 0.4769-0.4782; M1 stays at 243 with precision 1.00.
- **So the fine key defends against collisions only.** The flood is a movement-rule failure - one
  confirmation sends a whole family to the top - and a finer family is still a family.

### F3 - under the gate, feedback barely touches the future queue on this data

In [15]:
calibration, future = split["coarse"]
control = {f.alert_id: (cls, score) for cls, score, f in E.rank(future, {}, True, True)}
for formula in ("C1", "C2"):
    families, log, queue = trace(formula, "M1", 0.05, SEED, True)
    opened = {k: s for k, s in families.items() if gate_open(s)}
    moved = [(f, control[f.alert_id], (cls, score)) for cls, score, f in queue
             if (cls, score) != control[f.alert_id]]
    print(f"{formula} + M1, gated, coarse key, 5 % error, seed {SEED}:")
    print(f"   families that learned from at least one verdict: {len(families)}")
    print(f"   families whose gate is open: {len(opened)}")
    for family, state in opened.items():
        members = [f for f in future if f.family == family]
        effect = tuple(round(x, 2) for x in F.effective(state, True))
        print(f"      {family}: verdicts {verdicts(log, family)}; applied {effect}; "
              f"{len(members)} future flows, {sum(f.malicious for f in members)} attacks, "
              f"detection scores {min((f.detection_score for f in members), default='-')}"
              f"-{max((f.detection_score for f in members), default='-')}")
    kinds = Counter(("attack" if f.malicious else "benign",
                     "up" if (after[0], -after[1]) < (before[0], -before[1]) else "down")
                    for f, before, after in moved)
    print(f"   future flows whose queue position changed against the control: {len(moved)} {dict(kinds)}\n")

C1 + M1, gated, coarse key, 5 % error, seed 20260911:
   families that learned from at least one verdict: 34
   families whose gate is open: 4
      ('DDoS', 80, 'TCP', '-'): verdicts {('confirm_true_positive', 'correct', 'attack'): 116, ('mark_false_positive', 'wrong', 'attack'): 8}; applied (20.0, -5); 0 future flows, 0 attacks, detection scores ---
      ('Brute Force', 21, 'TCP', 'SIG-FTP-BRUTE-FORCE'): verdicts {('confirm_true_positive', 'correct', 'attack'): 89, ('mark_false_positive', 'wrong', 'attack'): 5}; applied (20.0, -5); 0 future flows, 0 attacks, detection scores ---
      ('Brute Force', 22, 'TCP', 'SIG-SSH-BRUTE-FORCE'): verdicts {('confirm_true_positive', 'correct', 'attack'): 30, ('mark_false_positive', 'wrong', 'attack'): 2}; applied (20.0, -5); 0 future flows, 0 attacks, detection scores ---
      ('Benign', 53, 'UDP', '-', '172.31.0.2'): verdicts {('mark_false_positive', 'correct', 'benign'): 16, ('confirm_true_positive', 'wrong', 'benign'): 3}; applied (-30.0, 4)

**Interpretation.**

- **Few families pass the gate, and they are not where the future is.** In the traced arm (5 % error,
  first seed) 4 of 34 learned families pass under C1, and 4 of 32 under C2.
- Three of the four are attack families - DDoS on port 80, FTP and SSH brute force - heavily confirmed,
  but **none has a single flow in the future half**. On a timestamp split those campaigns fall entirely
  in the calibration half, so their learning has nothing to reach.
- The fourth, the benign DNS family to 172.31.0.2, has 779 future flows, already at the bottom of the
  queue (scores 0.00-0.03). **C1** applies -30 there, so the 10 flows scoring above zero move down, and
  nothing else changes. **C2** applies nothing: Elo's step is zero when dismissing a score near zero, and
  its demotion has no lower class to move into.
- So on this data "no harm" also means "little effect". This is not a test of the efficiency claim; it
  shows the demo sample offers almost no recurring family for feedback to act on.
- **Correction.** An earlier draft said the passing attack families sat "at score 100 in the Tier 2
  band". They have no future members at all.

### F4 - the gate asks for more evidence, not the right evidence

A constructed case on the real future flows: suppose the analyst **correctly** dismisses three ML
false positives that fall into the coarse Web Attack family - three agreeing verdicts, so the gate
opens. The cell applies that family state with the experiment's own `rank` and reports what happens to
the true attacks in the family, then repeats it under the fine key, where the three dismissals are
keyed on AL-00478's destination.

In [16]:
params = F.RankingParams(formula="C1", movement="M1")
weight = chart.weight(WEB[0])


def three_dismissals():
    state = F.FamilyState()
    for _ in range(3):
        F.apply_verdict(state, params, "mark_false_positive", 99.89, weight)
    return state


state = three_dismissals()
print(f"C1 + M1, three correct dismissals at w = {weight}: gate open {gate_open(state)}, "
      f"applied (adjustment, offset) = {F.effective(state, True)}\n")
for key, family in (("coarse", WEB), ("fine", fine_of["AL-00478"])):
    future_k = split[key][1]
    before = {f.alert_id: (cls, score) for cls, score, f in E.rank(future_k, {}, True, True)}
    after = E.rank(future_k, {family: three_dismissals()}, True, True)
    hit = [(before[f.alert_id], (cls, score)) for cls, score, f in after
           if f.malicious and (cls, score) != before[f.alert_id]]
    left_tier2 = sum(1 for b, a in hit if b[0] == 0 and a[0] != 0)
    print(f"{key:<6} key, family {family}:")
    print(f"   true attacks moved: {len(hit)}, of which left the Tier 2 band: {left_tier2}")
    if hit:
        print(f"   e.g. (class, score) {hit[0][0]} -> {hit[0][1]}")

C1 + M1, three correct dismissals at w = 0.8: gate open True, applied (adjustment, offset) = (-17.999999999999993, 1)

coarse key, family ('Web Attack', 80, 'TCP', '-'):
   true attacks moved: 59, of which left the Tier 2 band: 59
   e.g. (class, score) (0, 100.0) -> (3, 82.0)
fine   key, family ('Web Attack', 80, 'TCP', '-', '64.150.178.87'):
   true attacks moved: 0, of which left the Tier 2 band: 0


**Interpretation.**

- The gate counts verdicts; it cannot tell whether they are about the same thing. Three **correct**
  dismissals of Web Attack false positives open the coarse family's gate (learning -18 and a one-class
  demotion) and push all 59 true attacks out of the Tier 2 band - class 0 at score 100 to the `ml_only`
  band at 82. Every verdict was right, so nothing in the gate objects.
- The fine key stops it here: the dismissals stay with AL-00478's destination, and no true attack moves.
- The demo data never produced this case - the traced arms never gave that family more than one
  verdict - so it is a constructed case, reported as one. **It is why the fine key stays on the table as
  defence in depth.**

### F5 - what sel-3 chose, and on what

In [17]:
selection = pd.DataFrame(results["selection"])
selection.insert(0, "rank", range(1, len(selection) + 1))
with pd.option_context("display.width", 220):
    print(f"sel-3 ranking, best first:\n")
    print(selection.to_string(index=False))

sel-3 ranking, best first:

 rank formula movement family_key  meets_q27  safe  attacks_demoted_under_error  tier2_precision  tier2_load  mean_attack_position  precision_at_100  class_changes
    1      C1       M2     coarse       True  True                            0              1.0         243                0.0828               1.0        13.0000
    2      C1       M2       fine       True  True                            0              1.0         243                0.0828               1.0        14.0000
    3      C1       M1     coarse       True  True                            0              1.0         243                0.0828               1.0        25.3333
    4      C1       M1       fine       True  True                            0              1.0         243                0.0828               1.0        30.3333
    5      C2       M2     coarse       True  True                            0              1.0         243                0.0829               1.0    

In [18]:
gated_arms = agg[agg.guardrails & agg.gated]
visible = ["precision_at_50", "precision_at_100", "precision_at_200", "mean_attack_position",
           "last_attack_position", "benign_in_top_100", "floor_violations", "attacks_demoted",
           "tier2_load", "tier2_precision"]
index = ["formula", "family_key", "error_rate"]
m1 = gated_arms[gated_arms.movement == "M1"].set_index(index).sort_index()
m2 = gated_arms[gated_arms.movement == "M2"].set_index(index).sort_index()
print(f"gated, guarded arms: M1 and M2 identical in every analyst-visible metric "
      f"({len(visible)} metrics, {len(m1)} arm pairs): {m1[visible].equals(m2[visible])}")
print("\nclass changes during calibration at 5 % error (internal family state), M1 vs M2:")
print(gated_arms[gated_arms.error_rate == 0.05].pivot_table(
    index=["formula", "family_key"], columns="movement", values="class_changes").round(2).to_string())
print("\nmean attack position at 5 % error, gated, by formula:")
print(gated_arms[gated_arms.error_rate == 0.05].groupby("formula").mean_attack_position
      .agg(["min", "max"]).to_string())

gated, guarded arms: M1 and M2 identical in every analyst-visible metric (10 metrics, 24 arm pairs): True

class changes during calibration at 5 % error (internal family state), M1 vs M2:
movement               M1     M2
formula family_key              
C0      coarse      25.33  13.00
        fine        30.33  14.00
C1      coarse      25.33  13.00
        fine        30.33  14.00
C2      coarse      26.00  16.33
        fine        31.00  17.33
C3      coarse      26.00  16.33
        fine        31.00  17.33

mean attack position at 5 % error, gated, by formula:
            min     max
formula                
C0       0.0828  0.0828
C1       0.0828  0.0828
C2       0.0829  0.0829
C3       0.0829  0.0829


**Interpretation.**

- **sel-3 names C1 + M2 + coarse, and how it got there matters more than the name.**
- **Formula.** Every gated arm ties on safety, attacks demoted (0) and Tier 2 precision (1.00), so the
  decision falls to mean attack position: C1 0.0828 against C2/C3 0.0829. On a 2,500-flow queue that is
  about a quarter of a position. C0 also scores 0.0828 but fails Q27 - its step ignores severity. C1 is
  chosen as much by that requirement and by simplicity as by measured performance, and C3 is again
  indistinguishable from C2.
- **Movement and key.** M2 over M1, and coarse over fine, were decided by class changes (13 against
  25.33 for C1). That metric counts each family's *internal* state during calibration, including changes
  the gate withholds from the queue. Every analyst-visible metric is **identical** for M1 and M2 across
  all 24 gated arm pairs, so the rule's pick of M2 rests on a number the analyst never sees.

---
## 5. Decision

In [19]:
top = results["selection"][0]
print(f"selection[0] of run {RUN_ID} (rule {config['selection_rule']}):")
for name, value in top.items():
    print(f"  {name:<30} {value}")

selection[0] of run 20260911T121013Z (rule sel-3):
  formula                        C1
  movement                       M2
  family_key                     coarse
  meets_q27                      True
  safe                           True
  attacks_demoted_under_error    0
  tier2_precision                1.0
  tier2_load                     243
  mean_attack_position           0.0828
  precision_at_100               1.0
  class_changes                  13


**Decisions (project lead, 2026-09-11):**

- **Q30 - the agreement gate is adopted for S7b.** F1's counterfactual shows it alone removes both of
  run 2's failure modes, and every gated arm matches the control on every analyst-visible metric.
- **Q29 - the formula is C1**, the severity-weighted step. sel-3 names it, it meets Q27, and its lead over
  C2/C3 is a quarter of a queue position (F5).
- **Movement M1 stands**, pending the project lead. sel-3's pick of M2 rests on internal class changes
  (F5), and M1 is the rule that cannot flood Tier 2 if the gate is ever relaxed (F2).
- **Family key: coarse**, as sel-3 names. The fine key is offered as defence in depth against collisions
  the gate cannot see (F4).

---
## 6. Limits and next experiment

- **The efficiency claim is still untested.** The detector already ranks nearly every attack at the top
  (control mean attack position 0.0829), and on a timestamp split the confirmed attack families do not
  recur in the future half (F3). Feedback has almost nothing to reorder here.
- **Next experiment:** a stress test with a weaker or drifting detector, so the top of the queue holds
  false positives to learn from (`docs/ranking-and-escalation-design.md` §8).
- **The 250,655-flow training sample was not run** - model predictions with SHAP exist only for the demo
  sample, and the experiment fuses from them.
- **The analyst is simulated** - ground truth flipped at random. Real analysts err in correlated ways.
- **The gate's thresholds are the collaborator's** (3 verdicts, 0.67). They were adopted, not tuned; no
  run varies them.